<a href="https://colab.research.google.com/github/Malki9/3601763_BD2/blob/main/3601763_BD2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Import Libraries

In [71]:
# Import the required libaries
import pandas as pd
import pandas as np
import json
import re

### Load the Dataset

In [73]:
# Create variables and load the .csv datasets
# Added on_bad_lines='skip' to handle formatting errors in the reviews file
products = pd.read_csv('/content/products.csv')
reviews = pd.read_csv('/content/reviews.csv')
users = pd.read_csv('/content/users.csv')

In [72]:
# Read the file and try to parse it line-by-line manually to avoid 'Trailing data' errors
products_list = []
with open('/content/jcpenney_products.json', 'r') as f:
    for line in f:
        line = line.strip()
        if line:
            try:
                products_list.append(json.loads(line))
            except json.JSONDecodeError:
                # If a single line fails, it might be the start/end of a standard array
                continue

if not products_list:
    # If line-by-line failed, try reading the whole file as one array
    with open('/content/jcpenney_products.json', 'r') as f:
        jcpenney_products = pd.DataFrame(json.load(f))
else:
    jcpenney_products = pd.DataFrame(products_list)

# Load reviewers (usually works fine with lines=True)
jcpenney_reviewers = pd.read_json('/content/jcpenney_reviewers.json', lines=True)

print('Data loaded successfully')

Data loaded successfully


In [74]:
jcpenney_reviewers.head(4)

,Username,DOB,State,Reviewed
0,bkpn1412,31.07.1983,Oregon,[cea76118f6a9110a893de2b7654319c0]
1,gqjs4414,27.07.1998,Massachusetts,[fa04fe6c0dd5189f54fe600838da43d3]
2,eehe1434,08.08.1950,Idaho,[]
3,hkxj1334,03.08.1969,Florida,"[f129b1803f447c2b1ce43508fb822810, 3b0c9bc0be6..."


In [63]:
jcpenney_products.head(4)

,uniq_id,sku,name_title,description,list_price,sale_price,category,category_tree,average_product_rating,product_url,product_image_urls,brand,total_number_reviews,Reviews,Bought With
0,b6c0b6bea69c722939585baeac73c13d,pp5006380337,Alfred Dunner® Essential Pull On Capri Pant,You'll return to our Alfred Dunner pull-on cap...,41.09,24.16,alfred dunner,jcpenney|women|alfred dunner,2.625,http://www.jcpenney.com/alfred-dunner-essentia...,http://s7d9.scene7.com/is/image/JCPenney/DP122...,Alfred Dunner,8,"[{'User': 'fsdv4141', 'Review': 'You never hav...","[898e42fe937a33e8ce5e900ca7a4d924, 8c02c262567..."
1,93e5272c51d8cce02597e3ce67b7ad0a,pp5006380337,Alfred Dunner® Essential Pull On Capri Pant,You'll return to our Alfred Dunner pull-on cap...,41.09,24.16,alfred dunner,jcpenney|women|alfred dunner,3.000,http://www.jcpenney.com/alfred-dunner-essentia...,http://s7d9.scene7.com/is/image/JCPenney/DP122...,Alfred Dunner,8,"[{'User': 'tpcu2211', 'Review': 'You never hav...","[bc9ab3406dcaa84a123b9da862e6367d, 18eb69e8fc2..."
2,013e320f2f2ec0cf5b3ff5418d688528,pp5006380337,Alfred Dunner® Essential Pull On Capri Pant,You'll return to our Alfred Dunner pull-on cap...,41.09,24.16,view all,jcpenney|women|view all,2.625,http://www.jcpenney.com/alfred-dunner-essentia...,http://s7d9.scene7.com/is/image/JCPenney/DP122...,Alfred Dunner,8,"[{'User': 'pcfg3234', 'Review': 'You never hav...","[3ce70f519a9cfdd85cdbdecd358e5347, b0295c96d2b..."
3,505e6633d81f2cb7400c0cfa0394c427,pp5006380337,Alfred Dunner® Essential Pull On Capri Pant,You'll return to our Alfred Dunner pull-on cap...,41.09,24.16,view all,jcpenney|women|view all,3.500,http://www.jcpenney.com/alfred-dunner-essentia...,http://s7d9.scene7.com/is/image/JCPenney/DP122...,Alfred Dunner,8,"[{'User': 'ngrq4411', 'Review': 'You never hav...","[efcd811edccbeb5e67eaa8ef0d991f7c, 7b2cc00171e..."


In [64]:
jcpenney_products.head(4)

,uniq_id,sku,name_title,description,list_price,sale_price,category,category_tree,average_product_rating,product_url,product_image_urls,brand,total_number_reviews,Reviews,Bought With
0,b6c0b6bea69c722939585baeac73c13d,pp5006380337,Alfred Dunner® Essential Pull On Capri Pant,You'll return to our Alfred Dunner pull-on cap...,41.09,24.16,alfred dunner,jcpenney|women|alfred dunner,2.625,http://www.jcpenney.com/alfred-dunner-essentia...,http://s7d9.scene7.com/is/image/JCPenney/DP122...,Alfred Dunner,8,"[{'User': 'fsdv4141', 'Review': 'You never hav...","[898e42fe937a33e8ce5e900ca7a4d924, 8c02c262567..."
1,93e5272c51d8cce02597e3ce67b7ad0a,pp5006380337,Alfred Dunner® Essential Pull On Capri Pant,You'll return to our Alfred Dunner pull-on cap...,41.09,24.16,alfred dunner,jcpenney|women|alfred dunner,3.000,http://www.jcpenney.com/alfred-dunner-essentia...,http://s7d9.scene7.com/is/image/JCPenney/DP122...,Alfred Dunner,8,"[{'User': 'tpcu2211', 'Review': 'You never hav...","[bc9ab3406dcaa84a123b9da862e6367d, 18eb69e8fc2..."
2,013e320f2f2ec0cf5b3ff5418d688528,pp5006380337,Alfred Dunner® Essential Pull On Capri Pant,You'll return to our Alfred Dunner pull-on cap...,41.09,24.16,view all,jcpenney|women|view all,2.625,http://www.jcpenney.com/alfred-dunner-essentia...,http://s7d9.scene7.com/is/image/JCPenney/DP122...,Alfred Dunner,8,"[{'User': 'pcfg3234', 'Review': 'You never hav...","[3ce70f519a9cfdd85cdbdecd358e5347, b0295c96d2b..."
3,505e6633d81f2cb7400c0cfa0394c427,pp5006380337,Alfred Dunner® Essential Pull On Capri Pant,You'll return to our Alfred Dunner pull-on cap...,41.09,24.16,view all,jcpenney|women|view all,3.500,http://www.jcpenney.com/alfred-dunner-essentia...,http://s7d9.scene7.com/is/image/JCPenney/DP122...,Alfred Dunner,8,"[{'User': 'ngrq4411', 'Review': 'You never hav...","[efcd811edccbeb5e67eaa8ef0d991f7c, 7b2cc00171e..."


In [65]:
# Checking whether the indexes are correct and no repeating indexes
jcpenney_reviewers.head(4)

,Username,DOB,State,Reviewed
0,bkpn1412,31.07.1983,Oregon,[cea76118f6a9110a893de2b7654319c0]
1,gqjs4414,27.07.1998,Massachusetts,[fa04fe6c0dd5189f54fe600838da43d3]
2,eehe1434,08.08.1950,Idaho,[]
3,hkxj1334,03.08.1969,Florida,"[f129b1803f447c2b1ce43508fb822810, 3b0c9bc0be6..."


In [66]:
reviews.head(4)

,Uniq_id,Username,Score,Review
0,b6c0b6bea69c722939585baeac73c13d,fsdv4141,2,You never have to worry about the fit...Alfred...
1,b6c0b6bea69c722939585baeac73c13d,krpz1113,1,Good quality fabric. Perfect fit. Washed very ...
2,b6c0b6bea69c722939585baeac73c13d,mbmg3241,2,I do not normally wear pants or capris that ha...
3,b6c0b6bea69c722939585baeac73c13d,zeqg1222,0,I love these capris! They fit true to size and...


In [67]:
products.head(4)

,Uniq_id,SKU,Name,Description,Price,Av_Score
0,b6c0b6bea69c722939585baeac73c13d,pp5006380337,Alfred Dunner® Essential Pull On Capri Pant,Youll return to our Alfred Dunner pull-on capr...,41.09,2.625
1,93e5272c51d8cce02597e3ce67b7ad0a,pp5006380337,Alfred Dunner® Essential Pull On Capri Pant,Youll return to our Alfred Dunner pull-on capr...,41.09,3.000
2,013e320f2f2ec0cf5b3ff5418d688528,pp5006380337,Alfred Dunner® Essential Pull On Capri Pant,Youll return to our Alfred Dunner pull-on capr...,41.09,2.625
3,505e6633d81f2cb7400c0cfa0394c427,pp5006380337,Alfred Dunner® Essential Pull On Capri Pant,Youll return to our Alfred Dunner pull-on capr...,41.09,3.500


In [68]:
users.head(4)

,Username,DOB,State
0,bkpn1412,31.07.1983,Oregon
1,gqjs4414,27.07.1998,Massachusetts
2,eehe1434,08.08.1950,Idaho
3,hkxj1334,03.08.1969,Florida


### Data Exploration



In [75]:
# check the dataframes basic information like the datatype
users.info()
jcpenney_products.info()
jcpenney_reviewers.info()
products.info()
reviews.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Username  5000 non-null   object
 1   DOB       5000 non-null   object
 2   State     5000 non-null   object
dtypes: object(3)
memory usage: 117.3+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7982 entries, 0 to 7981
Data columns (total 15 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   uniq_id                 7982 non-null   object 
 1   sku                     7982 non-null   object 
 2   name_title              7982 non-null   object 
 3   description             7982 non-null   object 
 4   list_price              7982 non-null   object 
 5   sale_price              7982 non-null   object 
 6   category                7982 non-null   object 
 7   category_tree           7982 non-null   object 
 8   average_product_rati

In [82]:
# Summary statistics
jcpenney_products.describe()


,average_product_rating,total_number_reviews
count,7982.000000,7982.000000
mean,2.988683,4.893886
std,0.911673,3.314284
min,1.000000,1.000000
25%,2.500000,2.000000
50%,3.000000,4.000000
75%,3.500000,8.000000
max,5.000000,23.000000


In [100]:
# Check for null values in each dataframe as a count
# Display in a tabular format
# Function to create the dataframe consist of column and sum
def null_summary(dataset):
    return pd.DataFrame({
        "column": dataset.columns,
        "sum": dataset.isnull().sum().values
    })

# Create summaries of each dataframe
users_null = null_summary(users)
jc_reviews_null = null_summary(jcpenney_reviewers)
jc_products_null = null_summary(jcpenney_products)
products_null = null_summary(products)
reviews_null = null_summary(reviews)

# Put all in dictionary
tables = {
    "users": users_null,
    "jcpenney_reviews": jc_reviews_null,
    "jc_products": jc_products_null,
    "products": products_null,
    "reviews": reviews_null
}

# Find maximum number of rows
max_rows = max(len(dataset) for dataset in tables.values())

# Pad smaller tables
for name, dataset in tables.items():
    if len(dataset) < max_rows:
        padding = pd.DataFrame({
            "column": [""] * (max_rows - len(dataset)),
            "sum": [""] * (max_rows - len(dataset))
        })
        tables[name] = pd.concat([dataset, padding], ignore_index=True)

# Create final aligned table
final_table = pd.concat(tables, axis=1)

final_table

users     jcpenney_reviews                 jc_products       \
      column sum           column sum                  column  sum   
0   Username   0         Username   0                 uniq_id    0   
1        DOB   0              DOB   0                     sku    0   
2      State   0            State   0              name_title    0   
3                        Reviewed   0             description  543   
4                                                  list_price    0   
5                                                  sale_price    0   
6                                                    category    0   
7                                               category_tree    0   
8                                      average_product_rating    0   
9                                                 product_url    0   
10                                         product_image_urls    0   
11                                                      brand    0   
12                                       total_number_reviews    0   
13                                                    Reviews    0   
14                                                Bought With    0   

       products         reviews      
         column   sum    column sum  
0       Uniq_id     0   Uniq_id   0  
1           SKU    67  Username   0  
2          Name     0     Score   0  
3   Description   543    Review   0  
4         Price  2166                
5      Av_Score     0                
6                                    
7                                    
8                                    
9                                    
10                                   
11                                   
12                                   
13                                   
14

In [98]:
products['SKU'] = products['SKU'].str.strip()
products['SKU'] = products['SKU'].replace(' ', pd.NA)

In [99]:
products['SKU'].isnull().sum()

np.int64(67)

In [92]:
# get the summary statistics of all the vaiables including categorical variables in all the dataset
jcpenney_products.describe(include='all')

# Count the categories of each sku
jcpenney_products['sku'].value_counts()

,count
sku,
,67
pp5006210554,18
pp5004560042,16
pp5006990582,15
pp5002781170,14
...,...
1afd48a,1
pp5003950738,1
pp5007010088,1


In [93]:
''' Sometimes the some attributes has nan which are considered to be string values and hence they are not couted as null. Need to eliminate those as well.
This happens for attributes which are catergorical'''
# Iterate through each dataset and identify in each column that nan values are there which are strings
datasets = [users, jcpenney_reviewers, jcpenney_products, products, reviews]
for dataset in datasets:
    for column in dataset.columns:
      if dataset[column].dtype == 'object'and dataset[column].str.contains('nan').any():
          # Replace the values with NaN
          dataset[column] = dataset[column].replace('', pd.NA)
          dataset[column] = dataset[column].replace(' ', pd.NA)
          dataset[column] = dataset[column].replace('nan', pd.NA)



In [ ]:
# The users act as the base dataset which is linked with the jcpenney_reviewers and reviews dataset
# Check the Username column in the users, jcpenny_reviews and reviews dataset follows the same standard
# Import the libary re
import re
# Define the pattern where the first 04 char are simple letters from a-z and the last for as int from 0-9
pattern = r'^[a-z]{4}[0-9]{4}$'
# Get the unique values in each of the dataset of username columns as a Dictionary
users_username = { 'users' : set(users['Username']), # convert the username column to set, eliminates the duplicates
                   'reviews': set(reviews['Username']),
                   'jcpenney_reviewers' : set(jcpenney_reviewers['Username'])
                   }

# Iterate through each value in the users_username
for key, value in users_username.items():
  invalid_found = False
  for username in value:
    if not re.fullmatch(pattern, username):
      print(f" Invalid username in {key}: {username}")
      invalid_found = True
  if not invalid_found: print(f" All usernames in {key} are in the particular standard")

 All usernames in users are in the particular standard
 All usernames in reviews are in the particular standard
 All usernames in jcpenney_reviewers are in the particular standard


In [37]:
# Iterate through each column in the jcpenney product and identify whether the product details (such as SKU) are same as the products using Unique_id
for column in products.columns:
  # Check the iterated column is uniq_id
  if column == 'Uniq_id':
    unmatch = False
    # Iterate through each value in the products
    for uniq_id in jcpenney_products['uniq_id']:
      if uniq_id not in products['Uniq_id'].values:
        print(f"Unique_id {uniq_id} not found in products")
        unmatch = True

    if not unmatch:
      print(f"All the Unique_id found in products are there in the Jcpenny_product.json file")

  elif column == 'sku':
    unmatch = False
    # Iterate through each value in the
    for product_name in products['SKU']:
      if product_name not in jcpenney_products['sku'].values:
        print(f"SKU {product_name} not found in jcpenney_products")
        unmatch = True

    if not unmatch:
      print(f"All the SKU found in jcpenney_products are there in the producsts csv")


All the Unique_id found in products are there in the Jcpenny_product.json file


In [36]:
for uniq_id in jcpenney_products['uniq_id']:

    # Check if uniq_id exists in products
    if uniq_id in products['Uniq_id'].values:

        # Get SKU from both datasets
        sku_products = products.loc[products['Uniq_id'] == uniq_id, 'SKU'].values[0]
        sku_jc = jcpenney_products.loc[jcpenney_products['uniq_id'] == uniq_id, 'sku'].values[0]

        # Compare SKU
        if sku_products != sku_jc:
            print(f"Mismatch for Unique_id {uniq_id}:")
            print(f"   products SKU = {sku_products}")
            print(f"   jcpenney SKU = {sku_jc}")

    else:
        print(f"Unique_id {uniq_id} not found in products")

Mismatch for Unique_id ada678a4523077b3bc62f9f418972507:
   products SKU = nan
   jcpenney SKU = 
Mismatch for Unique_id 97d2d2ce9f22fb3218f724324097d988:
   products SKU = nan
   jcpenney SKU = 
Mismatch for Unique_id 6d3177f25b852be96c858f2d85f37f4c:
   products SKU = nan
   jcpenney SKU = 
Mismatch for Unique_id 08daf20a62b797053019a926687abb53:
   products SKU = nan
   jcpenney SKU = 
Mismatch for Unique_id fc213145168faa94bae198f173d27796:
   products SKU = nan
   jcpenney SKU = 
Mismatch for Unique_id 43aa8a676fef64c914f46aaf34b8a0e2:
   products SKU = nan
   jcpenney SKU = 
Mismatch for Unique_id 7efc9da483e1d094b93ccdf9e9998241:
   products SKU = nan
   jcpenney SKU = 
Mismatch for Unique_id 0563348ce32c1a86bd897690ccb672d4:
   products SKU = nan
   jcpenney SKU = 
Mismatch for Unique_id e08e41d720d8ef75e71ad1f09962bc2b:
   products SKU = nan
   jcpenney SKU = 
Mismatch for Unique_id e10b8cd3fc58b5cee61127f106ca1a61:
   products SKU = nan
   jcpenney SKU = 
Mismatch for Unique_

In [ ]:
# Check Username is consitent over users dataframe and jcpenny_reviewers
# Define a dictionary variable and assign the columns as set
users_username = {
    'users' : set(users['Username']), # convert the username column to set, eliminates the duplicates
    'jcpenney_reviewers' : set(jcpenney_reviewers['Username'])
}

# Intersect to get the common values
username_common = set.intersection(*users_username.values())
print(f'The number of users common among the two dataset : {len(username_common)}')


The number of users common among the two dataset : 4999


All the users in user.csv exists in jcpenny_reviews


In [ ]:
# check the duplicated Username in the users

In [ ]:
# checking whether all the users in jcpenny_reviewers are there in users.csv
jcpenney_reviewers.Username.isin(users.Username).value_counts()

In [ ]:
# checking whether all the unique_id are matches with the products and reviews
products.Uniq_id.isin(jcpenney_products.uniq_id).value_counts()

In [ ]:
# Extracting the Uniq_id code which does not match
products[~jcpenney_products.uniq_id.isin(products.Uniq_id)]

In [ ]:
# combine the dataset user.csv and reviews with username
users = pd.merge(users, reviews, on='Username')


In [ ]:
users

In [ ]:
# Identifying the types of the Data in the dataframes .reviews
reviews.info()

In [ ]:
reviews

In [ ]:
# Converting the Score column/ data type to numeric
reviews['Score'] = pd.to_numeric(reviews['Score'], errors='coerce')

In [ ]:
reviews.describe()

In [ ]:
# Checking whether the average score in products given to unique_id is the same as the average of the scores in reviews


reviews.groupby('Uniq_id')['Score'].mean()

# Checking the mean of the scores in reviews with the Av_Score


In [50]:
# Extracting a record inJcpenny_products where the Uniq_id is 000bde718ab945188a364aab4d8bcfaa
record = products.loc[products['SKU'] ==  'nan']

In [51]:
record

,Uniq_id,SKU,Name,Description,Price,Av_Score


In [ ]:
# Now that jcpenney_products is restored, we can perform the merge
comparison = jcpenney_products.merge(products, left_on='uniq_id', right_on='Uniq_id', how='inner')



In [ ]:
comparison

In [ ]:
match_count = comparison['score_match'].value_counts()

In [ ]:
match_count

In [ ]:
# extracting the  Unique_id, average_product_rating, Av_Score columns of the unmathed scores
unmatched_scores = comparison[comparison['score_match'] == False][['uniq_id', 'average_product_rating', 'Av_Score', 'Uniq_id']]

In [ ]:
comparison.info()

In [ ]:
unmatched_scores

In [ ]:
type(jcpenney_products)

In [ ]:
jcpenney_products.astype(str).duplicated().sum()

In [ ]:
# Dropping the duplicate records correctly
# We remove inplace=True when assigning back to the variable to ensure it returns the new DataFrame
jcpenney_products = jcpenney_products.astype(str).drop_duplicates()